This notebook implements a highly efficient calorie prediction pipeline using XGBoost on CPU, tailored for the Kaggle Playground Series S5E5 dataset. The process begins by loading the training and test data and performing essential feature engineering. Specifically, it creates cross-product terms between key numerical features such as height, weight, heart rate, and body temperature, capturing important interaction effects. Two additional derived features, BMI (body mass index) and Intensity (heart rate divided by duration), are added to improve predictive power based on known physiological relationships.

The Sex column is label encoded and cast to a categorical type for proper handling during model training. The target variable, Calories, is log-transformed using np.log1p to align with the RMSLE evaluation metric and stabilize the variance. A 5-fold cross-validation loop is then used to train an XGBoost model with carefully tuned hyperparameters including a moderate learning rate, subsampling, and tree_method="hist" to ensure fast CPU training. Early stopping is employed to avoid overfitting.

Out-of-fold (OOF) predictions are collected to compute an accurate estimate of model generalization error using RMSLE, which is calculated both per fold and across the full dataset. After training, predictions on the test set are averaged across folds, inverse-transformed using expm1, and clipped between 1 and 314 to prevent unrealistic calorie estimates. Finally, the results are saved in a submission file ready for upload to Kaggle.

<font color = "DeepSkyBlue">**Table of Contents**

1. [Import libraries](#1)
2. [Load data](#2)
3. [Feature Engineering](#3)
4. [Add BMI & Intensity](#4)
5. [Label Encoding](#5)
6. [Prepare train/test matrices](#6)
7. [Cross-validation](#7)
8. [Final RMSLE](#8)
9. [Predictions and submission](#9)

<a id = "1"></a><br>
<font color = "DeepSkyBlue">**Import libraries**

This section imports the essential Python libraries required for the machine learning pipeline. It includes pandas and numpy for data manipulation, time for tracking execution duration, and scikit-learn modules for cross-validation, label encoding, and evaluation using the RMSLE metric. Lastly, it imports XGBRegressor from the XGBoost library to serve as the core predictive model.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error
from xgboost import XGBRegressor

<a id = "2"></a><br>
<font color = "DeepSkyBlue">**Load data**

This section loads the competition datasets into memory. It reads the training, test, and sample submission CSV files from the Kaggle input directory using pandas.read_csv, preparing them for subsequent preprocessing and model training steps.

In [ ]:
# Load data
train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")
submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")

<a id = "3"></a><br>
<font color = "DeepSkyBlue">**Feature Engineering**

This section performs feature engineering by generating cross-product terms between all pairs of selected numerical features (e.g., Weight × Duration, Age × Heart_Rate). A function is defined to create these interaction terms, which are then applied to both the training and test datasets. These new features help the model capture non-linear relationships between variables, potentially improving prediction accuracy.

In [ ]:
# Feature Engineering
numerical_features = ["Age", "Height", "Weight", "Duration", "Heart_Rate", "Body_Temp"]

def add_feature_cross_terms(df, numerical_features):
    df_new = df.copy()
    for i in range(len(numerical_features)):
        for j in range(i + 1, len(numerical_features)):
            cross_term_name = f"{numerical_features[i]}_x_{numerical_features[j]}"
            df_new[cross_term_name] = df_new[numerical_features[i]] * df_new[numerical_features[j]]
    return df_new

train = add_feature_cross_terms(train, numerical_features)
test = add_feature_cross_terms(test, numerical_features)

<a id = "4"></a><br>
<font color = "DeepSkyBlue">**Add BMI & Intensity**

This section adds two domain-specific features to both the training and test datasets. BMI (Body Mass Index) is calculated using height and weight to reflect body composition, while Intensity is computed as heart rate divided by workout duration to capture the physical effort level. These features are known to be strong predictors of calorie expenditure.

In [ ]:
# Add BMI & Intensity
train["BMI"] = train["Weight"] / (train["Height"] / 100) ** 2
test["BMI"] = test["Weight"] / (test["Height"] / 100) ** 2

train["Intensity"] = train["Heart_Rate"] / train["Duration"]
test["Intensity"] = test["Heart_Rate"] / test["Duration"]

<a id = "5"></a><br>
<font color = "DeepSkyBlue">**Label Encoding**

This section encodes the categorical Sex column into numerical format using LabelEncoder, converting categories like "Male" and "Female" into integers. After encoding, the column is explicitly cast to the categorical data type to ensure compatibility with the XGBoost model’s handling of categorical features.

In [ ]:
# Label Encoding
le = LabelEncoder()
train['Sex'] = le.fit_transform(train['Sex'])
test['Sex'] = le.transform(test['Sex'])

train["Sex"] = train["Sex"].astype("category")
test["Sex"] = test["Sex"].astype("category")

<a id = "6"></a><br>
<font color = "DeepSkyBlue">**Prepare train/test matrices**

This section prepares the data for model training and evaluation. It defines X as the feature matrix by dropping the id and target (Calories) columns, and transforms the target variable y using log1p to stabilize variance and align with RMSLE. The test feature matrix X_test is similarly prepared. A 5-fold cross-validation strategy is set up using KFold, and placeholder arrays are initialized to store out-of-fold (oof) predictions and averaged test predictions (pred).

In [ ]:
# Prepare train/test matrices
X = train.drop(columns=["id", "Calories"])
y = np.log1p(train["Calories"])
X_test = test.drop(columns=["id"])
FOLDS = 5

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=42)
oof = np.zeros(len(train))
pred = np.zeros(len(test))

<a id = "7"></a><br>
<font color = "DeepSkyBlue">**Cross-validation**

This section performs 5-fold cross-validation using the XGBoost regressor. For each fold, it splits the data into training and validation sets, trains the model with carefully chosen hyperparameters on the training set, and evaluates it on the validation set using the RMSLE metric (after reversing the log transformation with expm1). The model uses the CPU-optimized 'hist' tree method and includes early stopping to prevent overfitting. Predictions for the validation fold are stored in the oof array, and test predictions are accumulated across folds for later averaging. Execution time for each fold is also logged.

In [ ]:
# Cross-validation
for i, (train_idx, valid_idx) in enumerate(kf.split(X, y)):
    print(f"\n🔁 Fold {i+1}")
    
    x_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    x_valid, y_valid = X.iloc[valid_idx], y.iloc[valid_idx]

    start = time.time()
    
    model = XGBRegressor(
        tree_method="hist",  # CPU version
        predictor="auto",
        max_depth=10,
        colsample_bytree=0.7,
        subsample=0.9,
        n_estimators=2000,
        learning_rate=0.02,
        gamma=0.01,
        max_delta_step=2,
        early_stopping_rounds=100,
        eval_metric="rmse",
        enable_categorical=True,
        verbosity=0
    )

    model.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],
        verbose=100
    )

    oof[valid_idx] = model.predict(x_valid)
    pred += model.predict(X_test)

    fold_rmsle = np.sqrt(mean_squared_log_error(np.expm1(y_valid), np.expm1(oof[valid_idx])))
    print(f"📉 Fold {i+1} RMSLE: {fold_rmsle:.5f}")
    print(f"⏱️ Fold time: {time.time() - start:.1f} sec")

<a id = "8"></a><br>
<font color = "DeepSkyBlue">**Final RMSLE**

This section finalizes the model evaluation. It averages the test set predictions across all folds and calculates the overall RMSLE on the out-of-fold predictions by reversing the log transformation. This final RMSLE score provides an estimate of the model's expected performance on unseen data.

In [ ]:
# Final RMSLE
pred /= FOLDS
full_rmsle = np.sqrt(mean_squared_log_error(np.expm1(y), np.expm1(oof)))
print(f"\n✅ Final RMSLE: {full_rmsle:.5f}")

<a id = "9"></a><br>
<font color = "DeepSkyBlue">**Predictions and submission**

This section prepares the final predictions for submission. It reverses the log transformation on the averaged test predictions using expm1, then prints the mean and median values for inspection. Predictions are clipped between 1 and 314 to stay within realistic calorie bounds. Finally, the results are written to submission.csv in the required format for Kaggle submission.

In [ ]:
# Predictions and submission
y_preds = np.expm1(pred)
print('Mean prediction:', y_preds.mean())
print('Median prediction:', np.median(y_preds))

y_preds = np.clip(y_preds, 1, 314)
submission["Calories"] = y_preds
submission.to_csv("submission.csv", index=False)
submission.head()